# 12 — RAG: retrieval and answer functions (frozen interface)

**Purpose.** Build and freeze the two functions the rest of the project calls: `retrieve()` and `answer()`, exactly as specified in the notebook 08 contract. Everything before this notebook (09–11) only prepared data; this is the first notebook that answers a question.

- **Inputs:** newest `data/manifests/rag_index_*.json`, the Chroma collection and BM25 index it points to, `configs/rag.yaml` (`retrieval`).
- **Outputs:** the `retrieve()` and `answer()` functions (this notebook, imported or copied by whoever calls them), plus `data/manifests/rag_retrieval_smoke_<timestamp>.json` recording a smoke test.
- **No training, no evaluation.** Measuring hit@k / MRR / nDCG / citation precision / faithfulness against a real question set is the evaluator's notebook, not this one.

### For the evaluator — this is the interface, frozen

```text
retrieve(query: str, k: int = 5, before_date: str | None = None) -> list[dict]
    Each dict: chunk_id, doc_id, text, score, source_id, source_type,
               title, heading, language, published_effective, landing_url, licence

answer(question: str, forecast: dict, before_date: str | None = None, k: int = 5) -> dict
    {"answer": str, "cited_chunk_ids": list[str], "supported": bool}
```

- `before_date` (ISO `YYYY-MM-DD`) is the forecast origin quarter's end date. No returned chunk has `published_effective` after it. Passing `None` disables the cutoff — **never do this against a real forecast**; it exists only for exploring the corpus.
- `retrieve()` is deterministic: same query, same `before_date`, same index → same results, every time.
- `supported = False` means retrieval found nothing that cleared the relevance floor; `answer` then says so and cites nothing. This is a valid, expected outcome, not an error — do not count it as a bug on its own.
- **The answer generator in this notebook is a placeholder** (see the section below). It does not call an LLM. It exists so the interface, the citations and the "no supporting source" behaviour can be tested now; the prose itself does not yet reflect what a real fine-tuned explainer would write.

In [ ]:
# Mount Drive on Colab; skipped automatically when running locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
except ImportError:
    pass

In [ ]:
# Only needed on a fresh runtime; the project requirements already list these.
# !pip install -q sentence-transformers chromadb rank_bm25 snowballstemmer pyyaml pandas

In [ ]:
import os, re, json, pickle, hashlib, datetime as dt
from pathlib import Path
import yaml
import pandas as pd

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "rag.yaml").is_file():
            return cand
    return p

REPO = _find_repo()
MAN = REPO / "data" / "manifests"
CFG = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))
RCFG = CFG["retrieval"]
print("repo:", REPO)
print("retrieval config:", RCFG)

## Inputs check

Pin the index manifest, so retrieval says exactly which index it queried. `INDEX_MANIFEST = None` takes the newest one. Stops here, with a clear message, if the Chroma collection or the BM25 file do not match the manifest.

In [ ]:
INDEX_MANIFEST = None   # e.g. "rag_index_20260922T094625Z.json"

if INDEX_MANIFEST:
    index_manifest_path = MAN / INDEX_MANIFEST
else:
    found = sorted(MAN.glob("rag_index_*.json"))
    assert found, "No rag_index_*.json manifest found. Run notebook 11 first."
    index_manifest_path = found[-1]

index_manifest = json.loads(index_manifest_path.read_text())
chroma_dir = REPO / index_manifest["semantic"]["persist_dir"]
bm25_path = REPO / index_manifest["keyword"]["path"]
assert chroma_dir.is_dir(), f"{chroma_dir} not found. Run notebook 11 first."
actual_bm25_sha = hashlib.sha256(bm25_path.read_bytes()).hexdigest()
assert actual_bm25_sha == index_manifest["keyword"]["sha256"], "bm25_index.pkl does not match its manifest. Rerun notebook 11."

EMBEDDING_MODEL_ID = index_manifest["semantic"]["model_id"]
print("index manifest:", index_manifest_path.name)
print("embedding model:", EMBEDDING_MODEL_ID, "| chunks indexed:", index_manifest["semantic"]["count"])

## Load both indexes and the chunk metadata

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import snowballstemmer

client = chromadb.PersistentClient(path=str(chroma_dir))
collection = client.get_collection(index_manifest["semantic"]["collection_name"])
assert collection.count() == index_manifest["semantic"]["count"], "chroma collection changed since notebook 11 ran"

with open(bm25_path, "rb") as f:
    bm25_data = pickle.load(f)
bm25, bm25_chunk_ids = bm25_data["bm25"], bm25_data["chunk_ids"]

embedder = SentenceTransformer(EMBEDDING_MODEL_ID)
FI_STEMMER = snowballstemmer.stemmer("finnish")
WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)

# text and metadata for every chunk, keyed by chunk_id, for turning a hit into a full record
_all = collection.get(include=["documents", "metadatas"])
CHUNK_TABLE = {cid: {"text": doc, **meta} for cid, doc, meta in zip(_all["ids"], _all["documents"], _all["metadatas"])}
print("loaded", len(CHUNK_TABLE), "chunks from the index")

## The `retrieve()` function

Steps, matching the plan agreed earlier:

1. **Date filter first.** Chroma is queried with a `published_effective <= before_date` metadata filter; BM25 candidates outside the cutoff are masked out before ranking. Nothing later than `before_date` can ever appear.
2. **Retrieve `candidates_per_method` (20) from each side.**
3. **Each side drops its own weak matches** before fusion: semantic below `min_semantic_similarity`, BM25 below `min_bm25_score` (notebook 11 found that a BM25 score of 0 must not be treated as a match).
4. **Fuse the two survivor lists with Reciprocal Rank Fusion** (`rrf_k = 60`): a chunk found by both methods, even at different ranks, outranks one found by only one.
5. **Cap `max_chunks_per_doc` (2)** while walking the fused ranking, so one long bulletin cannot fill the whole result.
6. **Return the top `k`.** An empty list is a valid result and must be handled by the caller (`answer()` below does).

The query is stemmed for BM25 with **both its raw words and their Finnish stems**, since notebook 11 showed the stemmer does not always reduce a query to the same form as the document (`pitkäaikaistyöttömyys` vs `pitkäaikaistyöttömiä`); including both forms recovers some, not all, of that gap.

In [ ]:
def _bm25_query_tokens(query):
    words = WORD_RE.findall(query.lower())
    return list(dict.fromkeys(words + FI_STEMMER.stemWords(words)))   # raw + stemmed, deduplicated, order kept

def _date_to_epoch(date_str):
    return dt.date.fromisoformat(date_str).toordinal()

def _semantic_candidates(query, before_date, n):
    embedding = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    # Chroma's metadata filter only compares numbers, so the date cutoff goes through
    # the parallel `published_effective_epoch` integer field written by notebook 11.
    where = {"published_effective_epoch": {"$lte": _date_to_epoch(before_date)}} if before_date else None
    result = collection.query(query_embeddings=[embedding], n_results=n, where=where)
    hits = []
    for rank, (chunk_id, distance) in enumerate(zip(result["ids"][0], result["distances"][0])):
        similarity = 1 - distance   # Chroma's cosine "distance" in an hnsw:space=cosine collection
        if similarity >= RCFG["min_semantic_similarity"]:
            hits.append((chunk_id, rank, similarity))
    return hits

def _bm25_candidates(query, before_date, n):
    scores = bm25.get_scores(_bm25_query_tokens(query))
    eligible = [
        i for i, cid in enumerate(bm25_chunk_ids)
        if scores[i] >= RCFG["min_bm25_score"]
        and (not before_date or CHUNK_TABLE[cid]["published_effective"] <= before_date)
    ]
    eligible.sort(key=lambda i: -scores[i])
    return [(bm25_chunk_ids[i], rank, float(scores[i])) for rank, i in enumerate(eligible[:n])]

def retrieve(query, k=None, before_date=None):
    """See the frozen interface at the top of this notebook."""
    k = k or RCFG["default_k"]
    n = RCFG["candidates_per_method"]
    semantic_hits = _semantic_candidates(query, before_date, n)
    bm25_hits = _bm25_candidates(query, before_date, n)

    rrf_k = RCFG["rrf_k"]
    fused = {}   # chunk_id -> {"rrf": float, "semantic_score": float | None, "bm25_score": float | None}
    for chunk_id, rank, similarity in semantic_hits:
        fused.setdefault(chunk_id, {"rrf": 0.0, "semantic_score": None, "bm25_score": None})
        fused[chunk_id]["rrf"] += 1 / (rrf_k + rank)
        fused[chunk_id]["semantic_score"] = similarity
    for chunk_id, rank, score in bm25_hits:
        fused.setdefault(chunk_id, {"rrf": 0.0, "semantic_score": None, "bm25_score": None})
        fused[chunk_id]["rrf"] += 1 / (rrf_k + rank)
        fused[chunk_id]["bm25_score"] = score

    ranked = sorted(fused.items(), key=lambda kv: -kv[1]["rrf"])
    results, per_doc_count = [], {}
    for chunk_id, scores in ranked:
        record = CHUNK_TABLE[chunk_id]
        if per_doc_count.get(record["doc_id"], 0) >= RCFG["max_chunks_per_doc"]:
            continue
        per_doc_count[record["doc_id"]] = per_doc_count.get(record["doc_id"], 0) + 1
        results.append({"chunk_id": chunk_id, "doc_id": record["doc_id"], "text": record["text"],
                        "score": round(scores["rrf"], 5), "semantic_score": scores["semantic_score"],
                        "bm25_score": scores["bm25_score"], "source_id": record["source_id"],
                        "source_type": record["source_type"], "title": record["title"],
                        "heading": record["heading"] or None, "language": record["language"],
                        "published_effective": record["published_effective"],
                        "landing_url": record["landing_url"], "licence": record["licence"]})
        if len(results) == k:
            break
    return results

## Check `retrieve()`

Same queries notebook 11 used to sanity-check the raw indexes, now through the real function: with and without the date cutoff, and past a cutoff that should return nothing relevant.

In [ ]:
CHECKS = [
    ("Kuinka monta uutta avointa työpaikkaa ilmoitettiin?", None),
    ("how many new job vacancies were reported", None),
    ("pitkäaikaistyöttömyys", None),                                   # long-term unemployment
    ("Uusia avoimia työpaikkoja elokuussa 2024", "2024-08-31"),        # cutoff should allow this
    ("Uusia avoimia työpaikkoja elokuussa 2024", "2020-01-01"),        # cutoff should exclude it
]
for query, before_date in CHECKS:
    hits = retrieve(query, k=3, before_date=before_date)
    print(f"=== {query!r} (before_date={before_date})  ->  {len(hits)} hit(s)")
    for h in hits:
        print(f"  [{h['score']}] {h['chunk_id']} ({h['language']}, eff {h['published_effective']}): {h['text'][:140]}")

In [ ]:
# The date cutoff must never leak a later document.
for query, before_date in [("avoimet työpaikat", "2018-06-30"), ("jobs vacant", "2025-06-30")]:
    hits = retrieve(query, k=RCFG["candidates_per_method"], before_date=before_date)
    leaked = [h for h in hits if h["published_effective"] > before_date]
    assert not leaked, f"date cutoff leaked: {leaked}"
print("date cutoff check passed")

## The `answer()` function — a placeholder generator

**This does not call a language model.** It is a deterministic template: it states the forecast from the `forecast` record (never from retrieval, per the contract), and adds one or two short, clearly quoted, cited sentences from the retrieved passages, or says plainly that none were found. This is enough to test the interface, the citation behaviour and the "no supporting source" case end to end, without needing an LLM or GPU here.

**It is meant to be replaced.** Once the fine-tuner confirms whether a separate model writes the explanation, that model's generation replaces the templating inside this function; `retrieve()`, the forecast record shape, and the return shape of `answer()` do not change.

`forecast` follows the notebook 08 contract (section 2): `series_id`, `table_id`, `dimensions`, `origin_quarter`, `target_quarter`, `horizon_q`, `latest_value`, `predicted_value`.

In [ ]:
def _forecast_sentence(forecast):
    change = forecast["predicted_value"] - forecast["latest_value"]
    direction = "rise to" if change > 0 else ("fall to" if change < 0 else "stay at")
    dims = ", ".join(str(v) for v in forecast["dimensions"].values())
    return (f"For {dims} ({forecast['table_id']}), the forecast is that vacancies will {direction} "
            f"{forecast['predicted_value']:,.0f} by {forecast['target_quarter']} "
            f"({forecast['horizon_q']}-quarter horizon from {forecast['origin_quarter']}, "
            f"from {forecast['latest_value']:,.0f}).")

def answer(question, forecast, before_date=None, k=None):
    """See the frozen interface at the top of this notebook."""
    passages = retrieve(question, k=k, before_date=before_date)
    lines = [_forecast_sentence(forecast)]
    if not passages:
        lines.append("No supporting source was found in the corpus for this question.")
        return {"answer": " ".join(lines), "cited_chunk_ids": [], "supported": False}

    cited = []
    for i, passage in enumerate(passages[:2], start=1):
        quote = passage["text"][:220].rsplit(" ", 1)[0] + ("..." if len(passage["text"]) > 220 else "")
        lines.append(f'[{i}] {passage["title"]}: "{quote}" ({passage["landing_url"]})')
        cited.append(passage["chunk_id"])
    return {"answer": " ".join(lines), "cited_chunk_ids": cited, "supported": True}

## Check `answer()`

A stub forecast record in the notebook 08 shape (**not real model output** — the fine-tuner has not confirmed this layout yet), against a question the corpus should support and one it should not.

In [ ]:
STUB_FORECAST = {
    "series_id": "12tu__MK01__SSS__SSS", "table_id": "12tu",
    "dimensions": {"Alue": "MK01 Uusimaa", "Ammattiryhmä": "SSS Total"},
    "origin_quarter": "2025Q4", "target_quarter": "2026Q4", "horizon_q": 4,
    "latest_value": 36400, "predicted_value": 33500,
}

print(json.dumps(answer("Why are vacancies expected to fall in Uusimaa?", STUB_FORECAST,
                        before_date="2025-12-31"), indent=2, ensure_ascii=False))
print()
print(json.dumps(answer("What will happen to vacancies for a very obscure and unrelated question?",
                        STUB_FORECAST, before_date="2013-01-01"), indent=2, ensure_ascii=False))

## Checks

The build stops if any of these fail.

In [ ]:
result = answer("Why are vacancies expected to change?", STUB_FORECAST, before_date="2024-08-31")
assert set(result) == {"answer", "cited_chunk_ids", "supported"}
assert isinstance(result["supported"], bool)
assert result["supported"] == bool(result["cited_chunk_ids"])
assert str(STUB_FORECAST["predicted_value"]) not in "".join(h["text"] for h in retrieve("vacancies", k=5)), \
    "a retrieved passage happens to contain the forecast number -- coincidence, not a numbers-from-retrieval leak"

unsupported = answer("nonexistent quzzlewhat industry never mentioned anywhere", STUB_FORECAST, before_date="2013-01-01")
assert unsupported["supported"] is False and unsupported["cited_chunk_ids"] == []
print("all checks passed")

## Write a smoke-test manifest

Not an evaluation (that is the evaluator's), just a record that the frozen interface ran against the current index, for the handoff.

In [ ]:
NOW_UTC = dt.datetime.now(dt.timezone.utc)
manifest = {
    "name": "JobAI RAG retrieval smoke test",
    "created_utc": NOW_UTC.isoformat(),
    "index_manifest": {"path": str(index_manifest_path.relative_to(REPO)),
                       "sha256": hashlib.sha256(index_manifest_path.read_bytes()).hexdigest()},
    "retrieval_config": RCFG,
    "interface_frozen": True,
    "answer_generator": "placeholder_template (no LLM call); see notebook cell above",
    "checks_passed": ["date cutoff never leaks a later document", "answer() shape and supported/citations agree",
                      "unsupported question returns no citations"],
    "sample_queries": {q: len(retrieve(q, k=3, before_date=bd)) for q, bd in CHECKS},
}
out = MAN / f"rag_retrieval_smoke_{NOW_UTC.strftime('%Y%m%dT%H%M%SZ')}.json"
out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print("wrote:", out)

## Known limitations (as of 2026-09-22)

- **`answer()` does not call a language model.** It is a template so the interface can be tested now. Swapping in a real generator is the next step once the fine-tuner confirms the explainer model.
- **Thresholds are unvalidated.** `min_semantic_similarity` (0.30) and `min_bm25_score` (0.5) in `configs/rag.yaml` come from manually reading a handful of results, not from measured hit@k / precision. The evaluator's retrieval evaluation set should tune these.
- **No reranker**, per the notebook 08 decision; `reranker_enabled: false` is a switch, not yet wired to code.
- **Regional and occupation detail is usually missing**, because tables were not indexed (notebook 10). A question that can only be answered from a table will correctly come back `supported: False`; that is expected, not a retrieval bug.
- **The BM25 query expansion (raw + Finnish stem) is a partial fix**, not a general solution to the stemmer gap found in notebook 11.
- **`retrieve()` re-embeds the query on every call** (loads `bge-m3` once per notebook run). Fine for a smoke test; a real serving path should keep the model warm.

**Handoff:** the evaluator can now write her question set and metrics against `retrieve()` and `answer()` above. The fine-tuner's confirmed forecast record replaces `STUB_FORECAST`.